In [1]:
!pip install transformers datasets evaluate accelerate pyarrow scikit-learn -q

In [2]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

import evaluate

c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset_path = "../plantvillage_big.parquet"

df = pd.read_parquet(dataset_path)

print(df.head())

print("\nROWS:", len(df))

print("\nCOLUMNS:", df.columns)

                         caption                                         text
0  Pepper__bell___Bacterial_spot         pepper leaf yellow halo around spots
1  Pepper__bell___Bacterial_spot  pepper leaf black lesions on pepper foliage
2  Pepper__bell___Bacterial_spot         pepper leaf yellow halo around spots
3  Pepper__bell___Bacterial_spot  pepper leaf small dark spots on pepper leaf
4  Pepper__bell___Bacterial_spot  pepper leaf black lesions on pepper foliage

ROWS: 6000

COLUMNS: Index(['caption', 'text'], dtype='str')


In [4]:
df["text"] = df["text"].astype(str)

df = df.dropna()

df["text"] = df["text"].str.strip().str.lower()

df = df.sample(frac=1, random_state=42)

print(df.head())

                            caption  \
1782             Potato_Late_blight   
3917            Tomato_Early_blight   
221   Pepper__bell___Bacterial_spot   
2135             Tomato_Target_Spot   
5224      Tomato_Septoria_leaf_spot   

                                             text  
1782           potato leaf rotting potato foliage  
3917            tomato leaf early blight symptoms  
221   pepper leaf small dark spots on pepper leaf  
2135      tomato leaf infected tomato target spot  
5224    tomato leaf small circular septoria spots  


In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["caption"]
)

print("TRAIN:", len(train_df))
print("TEST:", len(test_df))

TRAIN: 4800
TEST: 1200


In [6]:
encoder = LabelEncoder()

train_df["label"] = encoder.fit_transform(train_df["caption"])

test_df["label"] = encoder.transform(test_df["caption"])

num_classes = len(encoder.classes_)

print("CLASSES:", num_classes)

print(encoder.classes_)

CLASSES: 15
['Pepper__bell___Bacterial_spot' 'Pepper__bell___healthy'
 'Potato_Early_blight' 'Potato_Late_blight' 'Potato_healthy'
 'Tomato_Bacterial_spot' 'Tomato_Early_blight' 'Tomato_Late_blight'
 'Tomato_Leaf_Mold' 'Tomato_Septoria_leaf_spot'
 'Tomato_Spider_mites_Two_spotted_spider_mite' 'Tomato_Target_Spot'
 'Tomato_Tomato_YellowLeaf_Curl_Virus' 'Tomato_Tomato_mosaic_virus'
 'Tomato_healthy']


In [7]:
train_dataset = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df.reset_index(drop=True)
)

print(train_dataset)

Dataset({
    features: ['caption', 'text', 'label'],
    num_rows: 4800
})


In [8]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding=True,
        max_length=64
    )

train_dataset = train_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 1200/1200 [00:00<00:00, 23759.39 examples/s]


In [9]:
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    dropout=0.2,
    attention_dropout=0.2
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2255.29it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return accuracy.compute(
        predictions=preds,
        references=labels
    )

In [11]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=5,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    weight_decay=0.01,

    warmup_ratio=0.1,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    logging_steps=50,

    save_total_limit=2,

    report_to="none"
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    data_collator=DataCollatorWithPadding(tokenizer),

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

In [13]:
trainer.train()

c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.352271,0.130860,1.000000
2,0.017580,0.008202,1.000000
3,0.008181,0.003911,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]
c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]
c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.63it/s]


TrainOutput(global_step=900, training_loss=0.5802116099993387, metrics={'train_runtime': 619.3274, 'train_samples_per_second': 38.752, 'train_steps_per_second': 2.422, 'total_flos': 40991602608000.0, 'train_loss': 0.5802116099993387, 'epoch': 3.0})

In [14]:
output = trainer.predict(test_dataset)

preds = np.argmax(output.predictions, axis=1)

pred_labels = encoder.inverse_transform(preds)

true_labels = encoder.inverse_transform(
    test_df["label"].values
)

for i in range(10):

    print("TEXT:", test_df["text"].iloc[i])

    print("TRUE:", true_labels[i])

    print("PRED:", pred_labels[i])

    print("-" * 50)

print("\nACCURACY:", output.metrics["test_accuracy"])

c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TEXT: pepper leaf fresh bell pepper foliage
TRUE: Pepper__bell___healthy
PRED: Pepper__bell___healthy
--------------------------------------------------
TEXT: pepper leaf healthy green pepper leaf
TRUE: Pepper__bell___healthy
PRED: Pepper__bell___healthy
--------------------------------------------------
TEXT: tomato leaf yellow stippling on tomato leaf
TRUE: Tomato_Spider_mites_Two_spotted_spider_mite
PRED: Tomato_Spider_mites_Two_spotted_spider_mite
--------------------------------------------------
TEXT: tomato leaf brown concentric lesions
TRUE: Tomato_Early_blight
PRED: Tomato_Early_blight
--------------------------------------------------
TEXT: tomato leaf septoria fungal disease
TRUE: Tomato_Septoria_leaf_spot
PRED: Tomato_Septoria_leaf_spot
--------------------------------------------------
TEXT: tomato leaf infected tomato target spot
TRUE: Tomato_Target_Spot
PRED: Tomato_Target_Spot
--------------------------------------------------
TEXT: potato leaf dry brown potato foliage


In [15]:
import os
import shutil

SAVE_DIR = "../distilbert_model"

# remove old folder
if os.path.exists(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)

os.makedirs(SAVE_DIR)

# save model
trainer.model.save_pretrained(
    SAVE_DIR,
    safe_serialization=False
)

# save tokenizer
tokenizer.save_pretrained(SAVE_DIR)

# save classes
with open(f"{SAVE_DIR}/classes.json", "w") as f:

    json.dump(
        encoder.classes_.tolist(),
        f
    )

print("✅ MODEL SAVED SUCCESSFULLY")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.30it/s]

✅ MODEL SAVED SUCCESSFULLY
